# distance preservation under super resolution
Evaluating how well the 3-D output embeddings preserve the 11-channel ERA5 feature-space distances when the model is asked to reconstruct a dense grid (super resolution): N_in=200 source nodes are observed, N_out=1000 target nodes have known positions but unknown features (their true features are only used for scoring).

1. load model (subgraph / cross-graph variant)
2. build per-date cross graphs: N_in=200 source nodes, N_out=1000 target nodes, 2019-01, full bbox
3. predict NOFE + 6 baselines (PCA, t-SNE, UMAP, PHATE, SpatialPCA, Concord) on the source features and carry the baseline embeddings to the target nodes
4. quantify feature-distance preservation: correlation + stress, global (all node pairs) and local (target graph edges)

---
### Utilities

In [ ]:
import os
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('NUMEXPR_NUM_THREADS', '1')
os.environ.setdefault('NUMBA_NUM_THREADS', '1')
import sys
import json
import random
import numpy as np
import pandas as pd
import torch

from scipy.interpolate import LinearNDInterpolator
from scipy.spatial import cKDTree
from pathlib import Path

project_root = Path.cwd().parent.parent
sys.path.append(str(project_root / 'src'))
sys.path.append(str(project_root / 'ERA5' / 'scripts'))

# Import utilities
from utilities import KernelNN, KernelNN_subgraph, setup_device

sys.path.append(f"{project_root}/ERA5/scripts/")
from utilities_era5 import ERA5_Graph_Generator as Data_Generator
data_path = project_root / 'ERA5' / 'data'

# Device setup
device = setup_device()
print(f'Using device: {device}')

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import subprocess
import phate


def load_model(id: str, model_type: str = 'random_crop'):
    model_path = project_root / 'ERA5' / 'trained_models' / model_type / f'{id}.pth'
    config_path = project_root / 'ERA5' / 'trained_models' / model_type / 'logs' / 'configs' / f'{id}.json'
    with open(config_path, 'r') as f:
        config = json.load(f)

    if model_type in ['random_crop', 'regular']:
        model = KernelNN(
            width=config['width'],
            ker_width=config['ker_width'],
            depth=config['depth'],
            ker_in=config['edge_attr'],
            in_width=config['node_features'],
            out_width=config['out_width'],
        ).to(device)
    elif model_type in ['subgraph', 'random_crop_subgraph']:
        model = KernelNN_subgraph(
            width=config['width'],  # number features per node after lifting
            ker_width=config['ker_width'],  # dimension of single kernel layer
            depth=config['depth'],  # number of graph layers / message passings
            ker_in=config['edge_attr'],  # number of edge features
            known_node_dim=config['input_node_features'],  # only needed when lifting is applied within model
            unknown_node_dim=config['output_node_features'],
            out_width=config['out_width'],
        ).to(device)
    else:
        raise ValueError(f'unknown model_type {model_type!r}')

    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    return model, config


# ERA5_Graph_Generator.generate_normed_sample() draws the random subset with
# np.random.choice, so the sampled graph -- and every embedding derived from it below --
# changes run-to-run unless the global RNGs are seeded first. The in-notebook methods
# pass random_state=42 explicitly, and the UMAP/concord runner scripts hardcode their own
# seed, so seeding here is what makes the sampled input itself reproducible.
SEED = 42


def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed()

# ---------------------------------------------------------------------------
# Shared prediction helpers.
#
# UMAP, Concord and SpatialPCA each only exist in their own conda env, so they are
# invoked as subprocesses. Buffer ids are per-process: with the fixed 'umap_buffer'
# names used by 2d_scatter.ipynb, two concurrently running notebooks race on the same
# .npy/.csv files.
METHOD_NAMES = ['nofe', 'pca', 'tsne', 'umap', 'phate', 'spatialpca', 'concord']
BASELINE_NAMES = METHOD_NAMES[1:]

_spatialpca_proc = None
_spatialpca_dir = f'{project_root}/ERA5/spatialpca'
_spatialpca_call_id = f'spatialpca_buffer_{os.getpid()}'


def _spatialpca_server():
    """Start the persistent SpatialPCA R server once; return the process.

    Library load (SpatialPCA pulls in Seurat, SPARK, Rcpp) costs ~tens of seconds, so
    the server loads it once and then answers one request per stdin line.
    """
    global _spatialpca_proc
    if _spatialpca_proc is not None and _spatialpca_proc.poll() is None:
        return _spatialpca_proc
    _spatialpca_proc = subprocess.Popen(
        ['conda', 'run', '-n', 'spatialpca', '--no-capture-output',
         'Rscript', 'execute_spatialpca_server.R'],
        cwd=_spatialpca_dir, stdin=subprocess.PIPE, stdout=subprocess.PIPE,
        stderr=subprocess.PIPE, text=True, bufsize=1,
        # R searches the *user* library (~/Library/R/<arch>/<ver>/library) before the
        # conda env's own library. CRAN-built packages there link the CRAN R framework
        # rather than conda's libR, and SpatialPCA's Rcpp .so then segfaults at dyn.load.
        env=dict(os.environ, R_LIBS_USER='/nonexistent'),
    )
    ready = _spatialpca_proc.stdout.readline().strip()
    if ready != 'READY':
        raise RuntimeError(
            f'spatialPCA server failed to start: {ready!r}\n'
            f'{_spatialpca_proc.stderr.read()}'
        )
    return _spatialpca_proc


def spatialpca_embed(X, coords, n_components=3):
    """SpatialPCA embedding of X (samples x features) located at coords (samples x d).

    `coords` carries the node positions SpatialPCA builds its kernel from. SpatialPCA
    returns min(n_components, rank) columns, and its SpatialPCnum is capped by the
    sample count, so the N_in=200 source graphs yield the 3 requested components.
    """
    proc = _spatialpca_server()
    np.savetxt(f'{_spatialpca_dir}/{_spatialpca_call_id}_X.csv', X, delimiter=',')
    np.savetxt(f'{_spatialpca_dir}/{_spatialpca_call_id}_coords.csv', coords, delimiter=',')

    proc.stdin.write(f'{_spatialpca_call_id} {n_components}\n')
    proc.stdin.flush()
    response = proc.stdout.readline().strip()

    os.remove(f'{_spatialpca_dir}/{_spatialpca_call_id}_X.csv')
    os.remove(f'{_spatialpca_dir}/{_spatialpca_call_id}_coords.csv')

    if response != 'OK':
        raise RuntimeError(f'spatialPCA failed for {_spatialpca_call_id}: {response}')

    Y = np.loadtxt(f'{_spatialpca_dir}/res_{_spatialpca_call_id}.csv',
                   delimiter=',', skiprows=1)
    os.remove(f'{_spatialpca_dir}/res_{_spatialpca_call_id}.csv')
    return Y


def spatialpca_shutdown():
    """Stop the R server (call after the last SpatialPCA embedding)."""
    global _spatialpca_proc
    if _spatialpca_proc is not None and _spatialpca_proc.poll() is None:
        _spatialpca_proc.stdin.write('QUIT\n')
        _spatialpca_proc.stdin.flush()
        _spatialpca_proc.wait(timeout=10)
    _spatialpca_proc = None


def external_embed(env, workdir, script, buffer_id, X):
    """Run `script` under conda env `env`: it reads <buffer_id>.npy, writes res_<buffer_id>.npy.

    check=True: a crashed child (missing package, OOM) must raise here, otherwise the
    np.load below silently picks up a res_<buffer_id>.npy left over from an earlier run.
    """
    np.save(f'{workdir}/{buffer_id}.npy', X)
    subprocess.run(f'cd {workdir} && conda run -n {env} python {script} {buffer_id}',
                   shell=True, check=True)
    Y = np.load(f'{workdir}/res_{buffer_id}.npy')
    os.remove(f'{workdir}/{buffer_id}.npy')
    os.remove(f'{workdir}/res_{buffer_id}.npy')
    return Y

----
### load model + dataset

In [ ]:
model_id = 'subgraph_tkernel_base2_d2_2026-04-25_t13-59-37'
model, config = load_model(id=model_id, model_type='subgraph')

In [ ]:
lat_min, lat_max = config['lat_min'], config['lat_max']
lon_min, lon_max = config['lon_min'], config['lon_max']
k_in, k_out, k_cross = config['k_in'], config['k_out'], config['k_cross']
perplexity = config['perplexity']
input_sampling_method = config['input_sampling_method']
output_sampling_method = config['output_sampling_method']
feature_mode = config['feature_mode']
interpolate_graph_features = config['interpolate_graph_features']
vars = config['vars']
bbox = lon_min, lat_min, lon_max, lat_max

In [ ]:
dates = pd.date_range("2019-01-01", "2019-01-31").strftime("%Y-%m-%d").tolist()

N_in = 200
N_out = 1000

Generator = Data_Generator(data_path)

# sample_graph draws with np.random.choice and nothing in the sampling path re-seeds, so
# seed the global RNGs once here. Seeding once (not per date) makes the 31 dates draw a
# reproducible but distinct subset sequence.
set_seed()

print('generate data')

graphs = []
for date in dates:
    Generator.load_data(bbox=bbox, vars=vars, filename=f'{date}.nc')
    in_graph = Generator.sample_graph(N=N_in, k=k_in, perplexity=perplexity,
                                      sampling_method=input_sampling_method, gen_feature_mode=feature_mode)
    out_graph = Generator.sample_graph(N=N_out, k=k_out, perplexity=perplexity,
                                       sampling_method=output_sampling_method, gen_feature_mode=feature_mode)
    cross_graph = Generator.connect_graphs(in_graph=in_graph, out_graph=out_graph, k_cross=k_cross)
    cross_graph['x'] = Generator.construct_output_node_features(
        cross_graph=cross_graph, interpolate=interpolate_graph_features, gen_feature_mode=feature_mode)
    graphs.append(cross_graph.to(device))

---
## Get Predictions

In [ ]:
def interpolate_to_target(embedding, sample):
    """Carry a source-node embedding (n_source, 3) onto the target nodes (n_target, 3).

    LinearNDInterpolator is undefined outside the source convex hull -- ~8% of target nodes
    at N_in=200/N_out=1000 (measured) -- and it returns NaN there. Those rows are filled
    with the embedding of the nearest source node rather than left at the origin, which
    would inject unrelated points into the embedding.
    """
    x_in = np.asarray(sample.source.meta.gps)
    x_out = np.asarray(sample.target.meta.gps)

    z_out = LinearNDInterpolator(x_in, embedding)(x_out)
    missing = np.isnan(z_out).any(axis=1)
    if missing.any():
        _, nearest = cKDTree(x_in).query(x_out[missing])
        z_out[missing] = embedding[nearest]
    return z_out, int(missing.sum())


def predict_all(sample):
    """{method: (n_target, 3) array} for every method in METHOD_NAMES.

    NOFE predicts the target nodes directly. Every baseline is reduced on the source
    features (n_source, n_features) and then carried to the target nodes, so all seven
    arrays have n_target rows and share one lat/lon table.
    """
    preds = {'nofe': model(sample).detach().cpu().numpy()}

    X = sample.source.normed_values.detach().cpu().numpy()
    coords = sample.source.normed_coords.detach().cpu().numpy()

    print('PCA ...', end='\r')
    source_embeddings = {'pca': PCA(n_components=3).fit_transform(X)}

    print('t-SNE ...', end='\r')
    source_embeddings['tsne'] = TSNE(
        n_components=3, perplexity=30, learning_rate='auto', random_state=42
    ).fit_transform(X)

    print('UMAP ...', end='\r')
    source_embeddings['umap'] = external_embed(
        'umap', f'{project_root}/ERA5/umap', 'execute_umap.py',
        f'sr_umap_buffer_{os.getpid()}', X,
    )

    print('PHATE ...', end='\r')
    source_embeddings['phate'] = phate.PHATE(
        n_components=3, knn=15, mds_solver='smacof', random_state=42, verbose=False
    ).fit_transform(X)

    print('spatialPCA ...', end='\r')
    source_embeddings['spatialpca'] = spatialpca_embed(X, coords)

    print('Concord ...', end='\r')
    source_embeddings['concord'] = external_embed(
        'concord', f'{project_root}/ERA5/concord', 'execute_concord.py',
        f'sr_concord_buffer_{os.getpid()}', X,
    )

    filled = {}
    for method in BASELINE_NAMES:
        preds[method], filled[method] = interpolate_to_target(source_embeddings[method], sample)
    print('out-of-hull target rows filled by nearest source node: '
          + ', '.join(f'{m}={filled[m]}' for m in BASELINE_NAMES))
    return preds

In [ ]:
pred_lists = {method: [] for method in METHOD_NAMES}

for i, sample in enumerate(graphs):
    print(f'{i + 1}/{len(graphs)}  {dates[i]}')
    preds = predict_all(sample)
    for method in METHOD_NAMES:
        pred_lists[method].append(preds[method])

spatialpca_shutdown()

---
## Quantitative Evaluation

In [ ]:
def pairwise_distances(X):
    # X: (N, d)
    norms = np.sum(X ** 2, axis=1, keepdims=True)  # (N, 1)
    dist2 = norms + norms.T - 2 * X @ X.T          # (N, N)
    dist2 = np.maximum(dist2, 0.0)                 # numerical stability: avoid tiny negatives
    return np.sqrt(dist2)


def distance_preservation(H, L):
    DH = pairwise_distances(H)
    DL = pairwise_distances(L)

    # upper triangle: avoid duplicates + diagonal
    i = np.triu_indices_from(DH, k=1)
    dH = DH[i]
    dL = DL[i]

    # correlation (1 = perfect preservation)
    return np.corrcoef(dH, dL)[0, 1]


def stress(H, L):
    DH = pairwise_distances(H)
    DL = pairwise_distances(L)

    i = np.triu_indices_from(DH, k=1)
    dH = DH[i]
    dL = DL[i]

    return np.sqrt(np.sum((dH - dL) ** 2) / np.sum(dH ** 2))


def edge_distances(X, edge_indices):
    # X: (N, d), edge_indices is torch_geometric's (2, E): row 0 sources, row 1 targets.
    return np.linalg.norm(X[edge_indices[0]] - X[edge_indices[1]], axis=1)


def edge_distance_preservation(H, L, edge_indices):
    return np.corrcoef(edge_distances(H, edge_indices), edge_distances(L, edge_indices))[0, 1]


def edge_stress(H, L, edge_indices):
    dH = edge_distances(H, edge_indices)
    dL = edge_distances(L, edge_indices)
    return np.sqrt(np.sum((dH - dL) ** 2) / np.sum(dH ** 2))

def stress_scale_free(H, L):
    """Kruskal stress-1 after rescaling L by its least-squares factor.

    `stress` above compares raw distances, so it also charges an embedding for living on a
    different scale than H. t-SNE is the case in point: sklearn's TSNE(learning_rate='auto')
    converges to a globally ~10x larger layout on 12 of the 31 dates here (std of the source
    embedding 22-28 instead of ~2.9), which alone pushes the raw stress of an otherwise
    mid-pack method (corr ~0.53) to 10-13. Dividing dL by the optimal factor
    sum(dH*dL)/sum(dL^2) removes that degree of freedom, so this column measures shape only
    and is comparable across methods. Every other method here sits near factor 1.3, where the
    two definitions agree to a few hundredths.
    """
    DH = pairwise_distances(H)
    DL = pairwise_distances(L)

    i = np.triu_indices_from(DH, k=1)
    dH = DH[i]
    dL = DL[i]

    scale = np.sum(dH * dL) / np.sum(dL ** 2)
    return np.sqrt(np.sum((dH - scale * dL) ** 2) / np.sum(dH ** 2))

def edge_stress_scale_free(H, L, edge_indices):
    """Kruskal stress-1 on the target graph edges, after rescaling L by its least-squares factor.

    The factor is fit on the edge subset rather than on all node pairs: kNN edges are
    short-range by construction, so a factor fitted over the full pairwise cloud is not
    the optimal fit here. Nothing else differs from `stress_scale_free`.
    """
    dH = edge_distances(H, edge_indices)
    dL = edge_distances(L, edge_indices)

    scale = np.sum(dH * dL) / np.sum(dL ** 2)
    return np.sqrt(np.sum((dH - scale * dL) ** 2) / np.sum(dH ** 2))

In [ ]:
score_names = ['corr (global)', 'stress (global)', 'corr (local)', 'stress (local)',
               'stress (global, scale-free)', 'stress (local, scale-free)']
per_date = {method: [] for method in METHOD_NAMES}  # method -> [(5 values), ...] one per date

for i in range(len(graphs)):
    # H: the true 11-channel features of the target nodes -- the super-resolution target
    # being reconstructed. graphs[i].target.edge_index is the target's own kNN graph
    # (k_out=5), so "local" means neighbourhoods in feature space of the target grid.
    H = graphs[i].target.meta.x_true.detach().cpu().numpy()
    edge_indices = graphs[i].target.edge_index.detach().cpu().numpy()

    for method in METHOD_NAMES:
        L = pred_lists[method][i]
        per_date[method].append([
            distance_preservation(H, L),
            stress(H, L),
            edge_distance_preservation(H, L, edge_indices),
            edge_stress(H, L, edge_indices),
            stress_scale_free(H, L),
            edge_stress_scale_free(H, L, edge_indices),
        ])

scores = pd.DataFrame(
    {method: [v for v in np.mean(per_date[method], axis=0)] for method in METHOD_NAMES},
    index=score_names,
)
scores_std = pd.DataFrame(
    {method: [v for v in np.std(per_date[method], axis=0)] for method in METHOD_NAMES},
    index=score_names,
)
scores

In [ ]:
print(', '.join(METHOD_NAMES))
for j, name in enumerate(score_names):
    print(f'{name}: ' + ' & '.join(
        f'{scores.loc[name, m]:.3f} $\\pm$ {scores_std.loc[name, m]:.3f}' for m in METHOD_NAMES
    ))